# Serialization

In [ ]:
import numpy as np
from laboneq.automation.serialization import load_automation_parameters_from_file
from laboneq.simple import *

from laboneq_applications.automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import (
    qubit_spectroscopy,
)
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

# Create a demonstration QuantumPlatform for a 4-qubit tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=4)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup (1x PQSC, 1x SHFQC, 1x HDAWG):
setup = qt_platform.setup

# And a 4-qubit tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of four LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

# We connect to the session in emulation mode:
session = Session(setup)
session.connect(do_emulation=True)

## Create folder store

In [ ]:
from pathlib import Path

folder_store = workflow.logbook.FolderStore(Path.cwd())

We disable saving in this tutorial. To enable it, simply run `folder_store.activate()`.

In [ ]:
folder_store.deactivate()

## Run the experiment in the automation

In [ ]:
auto = WorkflowAutomation(  # add uid parameter, integrate into workflow folder store
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters_from_file("serialization.yml"),
    name="example",
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs1",
    depends_on={"root"},
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs2",
    depends_on={"qs1"},
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs3",
    depends_on={"qs2"},
)

auto.add_layer(qs1)
auto.add_layer(qs2)
auto.add_layer(qs3)
auto.plot();

In [ ]:
auto.run()
auto.plot();

In [ ]:
auto.run_layer("qs2")

In [ ]:
auto.update_timestamp()

In [ ]:
auto.timestamp

## Basic serialization

In [ ]:
auto.automation_parameters

In [ ]:
auto.save_automation_parameters()

## Serialization after editing the layer parameters

In [ ]:
auto.get_layer("qs1").workflow_parameters

In [ ]:
auto.get_layer("qs1").workflow_parameters["q0"]["frequencies"] = np.linspace(1, 3, 3)

In [ ]:
auto.get_layer("qs1").workflow_parameters

In [ ]:
auto.save_automation_parameters()

## Deserialization (loading from a file)

In [ ]:
auto.automation_parameters

In [ ]:
auto.automation_parameters = None

In [ ]:
auto.load_automation_parameters("serialization.yml")

In [ ]:
auto.automation_parameters